# Standardised Feature Engineering Notebook

This notebook **standardises** the **feature engineering** workflow for **historical stablecoin datasets**. Users should **update** the file paths for both *data input and output* before running the notebook. The cleaned dataset is saved in **Parquet** format to *preserve variable types* and ensure consistency across downstream analysis. This workflow is designed to be easily replicated for other stablecoins by changing the relevant file paths only.

The predictive features are constructed to closely follow the framework proposed in Lee, Chiu, and Hsieh (2025), **Stablecoin depegging risk prediction**, published in the Pacific-Basin Finance Journal. This helps maintain consistency with the literature while allowing the feature engineering process to be applied systematically across different stablecoin datasets.

In [15]:
import pandas as pd
import numpy as np

In [55]:
stablecoin = "ust" # or "usdc", "usdt", "pax", "dai", "ust"
df = pd.read_parquet(f"clean_data/{stablecoin}_clean.parquet")
df.head()

,Unnamed: 0,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,symbol,timestamp,...,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,0,0,0,0,0,0,0,0,UST,2020-11-25 23:59:59,...,2020-11-25 23:59:59,2020-11-25 11:48:07,2020-11-25 17:19:07,1.010488,1.010752,0.993542,0.999572,3758726.64,1.332804e+07,13333749.41
1,1,0,0,0,0,0,0,0,UST,2020-11-26 23:59:59,...,2020-11-26 23:59:59,2020-11-26 03:33:07,2020-11-26 09:47:07,0.999360,1.022774,0.982691,1.000441,16210369.84,1.564638e+07,15639476.02
2,2,0,0,0,0,0,0,0,UST,2020-11-27 23:59:59,...,2020-11-27 23:59:59,2020-11-27 11:33:07,2020-11-27 16:07:07,1.000106,1.003605,0.994698,0.999515,4086972.72,1.743930e+07,17447758.34
3,3,0,0,0,0,0,0,0,UST,2020-11-28 23:59:59,...,2020-11-28 23:59:59,2020-11-28 19:25:11,2020-11-28 16:04:08,0.999476,1.004033,0.995601,0.999325,3721757.38,1.743599e+07,17447757.88
4,4,0,0,0,0,0,0,0,UST,2020-11-29 23:59:59,...,2020-11-29 23:59:59,2020-11-29 16:06:07,2020-11-29 21:04:07,0.999246,1.001828,0.997059,0.999641,856953.09,1.743870e+07,17444966.35


## Feature Engineering

### Rate of Change in Trading Price and Volume

In [56]:
# price percent changes
df["percent_change_24h"] = df["close"].pct_change(1) * 100
df["percent_change_7d"]  = df["close"].pct_change(7) * 100
df["percent_change_30d"] = df["close"].pct_change(30) * 100

# volume percent changes
df["volume_percent_change_24h"] = df["volume"].pct_change(1) * 100
df["volume_percent_change_7d"]  = df["volume"].pct_change(7) * 100
df["volume_percent_change_30d"] = df["volume"].pct_change(30) * 100

### Rate of Change in Market Information

In [57]:
def ln_past_over_today(x, k):
    return np.log(x.shift(k) / x)

# market cap log-changes
df["market_cap_percent_change_24h"] = ln_past_over_today(df["marketCap"], 1)
df["market_cap_percent_change_7d"]  = ln_past_over_today(df["marketCap"], 7)
df["market_cap_percent_change_30d"] = ln_past_over_today(df["marketCap"], 30)

# circulating supply log-changes
df["circulating_supply_percent_change_24h"] = ln_past_over_today(df["circulatingSupply"], 1)
df["circulating_supply_percent_change_7d"]  = ln_past_over_today(df["circulatingSupply"], 7)
df["circulating_supply_percent_change_30d"] = ln_past_over_today(df["circulatingSupply"], 30)

### Volatility Indicators

In [58]:
# avoid log(0) issues
eps = 1e-12
O = df["open"].clip(lower=eps)
H = df["high"].clip(lower=eps)
L = df["low"].clip(lower=eps)
C = df["close"].clip(lower=eps)

df["realized_daily_volatility"] = np.sqrt(
    (np.log(H / O) * np.log(H / C)) + (np.log(L / O) * np.log(L / C))
)

In [59]:
df["peg_error"] = df["close"] - 1
df["abs_peg_error"] = df["peg_error"].abs()

df["price_deviation_5d"]  = np.sqrt((df["peg_error"]**2).rolling(5).mean())
df["price_deviation_30d"] = np.sqrt((df["peg_error"]**2).rolling(30).mean())

downside = np.minimum(df["peg_error"], 0.0)
df["downward_price_deviation_5d"]  = np.sqrt((downside**2).rolling(5).mean())
df["downward_price_deviation_30d"] = np.sqrt((downside**2).rolling(30).mean())


### Cleanup
To ensure model compatibility, infinite values arising from logarithmic transformations and ratio-based features are first replaced with missing values. Observations containing missing values—primarily introduced by rolling-window and lagged feature construction—are subsequently removed. This ensures that all downstream analyses, including PCA and machine learning models, are performed on a complete and consistent dataset.

In [60]:
# replace inf/-inf from logs with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# drop NaNs (ie for early rows due to rolling/shift)
df = df.dropna().reset_index(drop=True)
df.head()

,Unnamed: 0,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,symbol,timestamp,...,circulating_supply_percent_change_24h,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d
0,30,0,0,0,0,0,0,0,UST,2020-12-25 23:59:59,...,-0.000116,-0.083712,-2.506935,0.004778,-0.001375,0.001375,0.002765,0.001560,0.002765,0.001541
1,31,0,0,0,0,0,0,0,UST,2020-12-26 23:59:59,...,0.000041,-0.083304,-2.347393,0.007252,-0.002121,0.002121,0.002795,0.001605,0.002795,0.001589
2,32,0,0,0,0,0,0,0,UST,2020-12-27 23:59:59,...,-0.000985,-0.084124,-2.238965,0.008277,-0.000279,0.000279,0.002279,0.001604,0.002279,0.001587
3,33,0,0,0,0,0,0,0,UST,2020-12-28 23:59:59,...,-0.017737,-0.089454,-2.256702,0.004996,-0.001755,0.001755,0.002359,0.001631,0.002359,0.001615
4,34,0,1,1,1,1,1,1,UST,2020-12-29 23:59:59,...,-0.037969,-0.126041,-2.294831,0.080241,-0.002543,0.002543,0.001790,0.001694,0.001790,0.001679


## Combining Sentiment Indicators

In [61]:
# create a daily date key (so merges are clean)
df["date"] = df["timestamp"].dt.date
df["date"] = pd.to_datetime(df["date"])
df.head()

,Unnamed: 0,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,symbol,timestamp,...,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,date
0,30,0,0,0,0,0,0,0,UST,2020-12-25 23:59:59,...,-0.083712,-2.506935,0.004778,-0.001375,0.001375,0.002765,0.001560,0.002765,0.001541,2020-12-25
1,31,0,0,0,0,0,0,0,UST,2020-12-26 23:59:59,...,-0.083304,-2.347393,0.007252,-0.002121,0.002121,0.002795,0.001605,0.002795,0.001589,2020-12-26
2,32,0,0,0,0,0,0,0,UST,2020-12-27 23:59:59,...,-0.084124,-2.238965,0.008277,-0.000279,0.000279,0.002279,0.001604,0.002279,0.001587,2020-12-27
3,33,0,0,0,0,0,0,0,UST,2020-12-28 23:59:59,...,-0.089454,-2.256702,0.004996,-0.001755,0.001755,0.002359,0.001631,0.002359,0.001615,2020-12-28
4,34,0,1,1,1,1,1,1,UST,2020-12-29 23:59:59,...,-0.126041,-2.294831,0.080241,-0.002543,0.002543,0.001790,0.001694,0.001790,0.001679,2020-12-29


### Load Fear & Greed

In [62]:
fg = pd.read_csv("raw_data/fear_greed_index.csv")

fg["date"] = pd.to_datetime(fg["date"], errors="coerce")  
fg = fg.dropna(subset=["date"]).sort_values("date")
fg["date"] = fg["date"].dt.tz_localize(None).dt.normalize()

# rename columns for clarity
fg = fg.rename(columns={
    "value": "fear_greed_index",
    "value_classification": "fear_greed_index_category",
    "date": "date"
})

### Load Fed Funds Rate

In [63]:
ff = pd.read_csv("raw_data/fed_funds_rate.csv")

ff["date"] = pd.to_datetime(ff["date"], errors="coerce")
ff = ff.dropna(subset=["date"]).sort_values("date")
ff["date"] = ff["date"].dt.tz_localize(None).dt.normalize()

### Merge

In [64]:
print(df["date"].dtype)
print(fg["date"].dtype)
print(ff["date"].dtype)

datetime64[s]
datetime64[us]
datetime64[us]


In [65]:
final_df = df.merge(fg, on="date", how="left").merge(ff, on="date", how="left")
final_df = final_df.drop(columns=["date"])
final_df.head()

,Unnamed: 0,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,symbol,timestamp,...,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,fear_greed_index,fear_greed_index_category,fed_funds_rate
0,30,0,0,0,0,0,0,0,UST,2020-12-25 23:59:59,...,0.004778,-0.001375,0.001375,0.002765,0.001560,0.002765,0.001541,94,Extreme Greed,0.09
1,31,0,0,0,0,0,0,0,UST,2020-12-26 23:59:59,...,0.007252,-0.002121,0.002121,0.002795,0.001605,0.002795,0.001589,93,Extreme Greed,0.09
2,32,0,0,0,0,0,0,0,UST,2020-12-27 23:59:59,...,0.008277,-0.000279,0.000279,0.002279,0.001604,0.002279,0.001587,91,Extreme Greed,0.09
3,33,0,0,0,0,0,0,0,UST,2020-12-28 23:59:59,...,0.004996,-0.001755,0.001755,0.002359,0.001631,0.002359,0.001615,92,Extreme Greed,0.09
4,34,0,1,1,1,1,1,1,UST,2020-12-29 23:59:59,...,0.080241,-0.002543,0.002543,0.001790,0.001694,0.001790,0.001679,91,Extreme Greed,0.09


---

## Save

In [66]:
print(final_df.shape)
final_df.dtypes

(502, 43)


Unnamed: 0                                        int64
depeg                                             int64
depeg_future_1d                                   int64
depeg_future_3d                                   int64
depeg_future_5d                                   int64
depeg_future_7d                                   int64
depeg_future_14d                                  int64
depeg_future_30d                                  int64
symbol                                              str
timestamp                                datetime64[us]
timeOpen                                 datetime64[us]
timeClose                                datetime64[us]
timeHigh                                 datetime64[us]
timeLow                                  datetime64[us]
open                                            float64
high                                            float64
low                                             float64
close                                           

In [67]:
final_df.to_parquet(f"clean_data/{stablecoin}_final.parquet")